# GHCN-Daily Climate Analysis with Apache Spark

**DATA420-26S2 (C) - Assignment 1 - GHCN Data Analysis using Spark**

This notebook is the single authoritative walkthrough of the analysis. It reads the Global
Historical Climatology Network daily summaries (~13 GB compressed, ~3.19 billion rows) from
Azure Blob Storage over `wasbs://`, builds an enriched station table, answers the assignment
questions, and produces the visualisations - orchestrating the reusable functions in the
[`ghcn`](../src/ghcn) package rather than repeating logic inline.

> **How to read this notebook.** The code cells are runnable **on the UC MADS Kubernetes Spark
> cluster** (they need the cluster and the read-only `campus-data` container; they cannot run on a
> laptop). Cell outputs are intentionally **not committed** here to keep the notebook clean and
> the results unambiguous. Each section states the key figures under a **Result** note, and the
> fully rendered, cluster-executed notebooks are preserved as evidence under
> [`docs/executed_notebooks/`](../docs/executed_notebooks). The full write-up with figures is in
> [`reports/final_report.md`](../reports/final_report.md).

**Sections:** Setup - Processing Q1-Q4 - Analysis Q1-Q3 - Visualizations Q1-Q2.

## 0 - Environment and Spark session

The SAS token for the output container is read from the environment (`AZURE_USER_SAS_TOKEN`); no secret is stored in the notebook or the package.

In [ ]:
import os
from ghcn.config.paths import default_paths
from ghcn.utils.spark_utils import build_spark_session

paths = default_paths()                     # resolves your cluster username
print("User output root:", paths.user_root)
print("GHCN data root:   ", paths.ghcnd_root)

# 4 executors x 2 cores is the assignment ceiling; used for the heavy scans below.
spark, sc = build_spark_session(
    executor_instances=4, executor_cores=2, worker_memory=4, master_memory=4,
)
print("Spark:", spark.version)

## Processing Q1 - Structure and size of the data

The `ghcnd` directory holds one gzip-compressed CSV per year under `daily/`, plus four
uncompressed fixed-width metadata files. Gzip is **not splittable**, so each yearly file is read
by a single task; the metadata is fixed-width, so it cannot use `spark.read.csv`.

> **Result (cluster run).** `daily` is **13.15 GiB compressed** (14,116,195,511 bytes) =
> **99.67%** of the 13.19 GiB total. It spans **265 year-files, 1750-2026** (1751-1762 missing).
> Streaming the 2025 file through `gzip -dc | wc -c` gives a **7.87x** expansion ratio, so the
> full uncompressed size is an estimated **~111 GB (103.5 GiB)**.

In [ ]:
# Explored with the hdfs CLI (no data loaded into Spark). Sizes captured from `hdfs dfs -du`.
from ghcn.utils.io_utils import bytes_to_human

DAILY_COMPRESSED_BYTES = 14_116_195_511
TOTAL_BYTES = DAILY_COMPRESSED_BYTES + 35_991_182 + 11_395_086 + 3_659 + 1_086
print("daily compressed:", bytes_to_human(DAILY_COMPRESSED_BYTES))
print("ghcnd total:     ", bytes_to_human(TOTAL_BYTES))
print(f"daily share:      {DAILY_COMPRESSED_BYTES / TOTAL_BYTES:.2%}")

# Uncompressed estimate from one representative year (2025).
ratio = 1_261_472_066 / 160_283_428
print(f"expansion ratio:  {ratio:.2f}x  ->  ~{DAILY_COMPRESSED_BYTES * ratio / 1e9:.0f} GB uncompressed")

## Processing Q2 - Loading metadata and defining the daily schema

Fixed-width tables are parsed with `spark.read.text` + `substring` via `ghcn.loaders`;
column ranges live in `ghcn.schemas`.

> **Schema gotcha.** Declaring `DATE` as `DateType` and reading with `spark.read.csv` returns
> **all-NULL dates** - Spark accepts the schema but will not parse the compact `YYYYMMDD` form.
> The fix is to read `DATE` as a string and convert with `to_date(col, "yyyyMMdd")`. `VALUE` is a
> double; `OBSERVATION_TIME` stays a string (an `HHMM` clock reading).
>
> **Result (cluster run).** Row counts - stations **132,501**, countries **219**, states **74**,
> inventory **782,417**; `daily` 2026 **17,988,944** rows; all of `daily` **3,190,506,382** rows.

In [ ]:
from ghcn.loaders.load_metadata import load_all_metadata
from ghcn.loaders.load_daily import load_daily, load_daily_year

meta = load_all_metadata(spark, paths)          # stations, countries, states, inventory
for name, df in meta.items():
    print(f"{name:10s}: {df.count():>9,} rows")

daily_2026 = load_daily_year(spark, paths, 2026)
daily = load_daily(spark, paths)                 # all 265 years, lazy; never collected
print("daily 2026:", daily_2026.count())
print("daily all: ", daily.count())

## Processing Q3 - The enriched stations table

Extract the 2-char country code from the station ID, LEFT JOIN `countries` and `states`, then
join per-station inventory metrics (element set, active span, core/other counts). Saved as
**Parquet partitioned by `COUNTRY_CODE`** so downstream reads prune by country and the `ELEMENTS`
array survives the round-trip (CSV cannot store it).

> **Result (cluster run).** Enriched table has **132,501 rows** (one per station). Of these,
> **37,422** are active in 2026 by inventory, **20,505** record all five core elements, and
> **16,189** record precipitation only. **16 countries** have stations with state codes
> (United States, Canada, Russia, several US/Pacific territories, and the Bahamas). 64 stations
> have no inventory record; 0 inventory records reference an unknown station.

In [ ]:
from ghcn.processing.enrich_stations import (
    build_enriched_stations, count_active_in_year, count_all_core, count_precip_only,
    summarise_state_countries,
)
from ghcn.processing.storage import save_parquet

enriched = build_enriched_stations(
    meta["stations"], meta["countries"], meta["states"], meta["inventory"],
)
print("rows:", enriched.count())
print("active 2026 (inventory):", count_active_in_year(enriched, 2026))
print("all five core elements: ", count_all_core(enriched))
print("precipitation only:     ", count_precip_only(enriched))
print("countries with state codes:", len(summarise_state_countries(enriched)))

save_parquet(enriched.repartition(3, "COUNTRY_CODE"),
             paths.output("enriched_stations"), partition_by="COUNTRY_CODE")

## Processing Q4 - Stations missing from `daily`

A full `daily`-`stations` join would shuffle 3.19 B rows. Instead: project `daily` to `ID`,
**broadcast** the 132k-row station set (LEFT SEMI), deduplicate, then LEFT ANTI against
`stations`. The physical plan confirms a `BroadcastHashJoin`, not a `SortMergeJoin`.

> **Result (cluster run).** **63 stations** appear in the metadata but never in `daily`;
> **0** daily station IDs are missing from the metadata.

In [ ]:
from ghcn.processing.validation import (
    stations_missing_from_daily, daily_ids_missing_from_stations,
)
print("stations never in daily:  ", stations_missing_from_daily(enriched, daily).count())
print("daily IDs not in stations:", daily_ids_missing_from_stations(enriched, daily).count())

## Analysis Q1 - Station counts

Total, network membership (GSN / HCN / CRN, with overlap counted explicitly), hemisphere,
US territories, and New Zealand.

> **Result (cluster run).** Total **132,501**; **GSN 991**, **HCN 1,218**, **CRN 234**, with
> **15** stations in more than one network; **25,357** in the Southern Hemisphere (latitude < 0);
> **442** in US territories (excluding the mainland); **New Zealand 15**.

In [ ]:
from ghcn.analysis.station_analysis import (
    total_stations, network_counts, southern_hemisphere_count,
    us_territory_count, new_zealand_count,
)
print("total:      ", total_stations(enriched))
print("networks:   ", network_counts(enriched))   # dict incl. IN_MORE_THAN_ONE
print("southern:   ", southern_hemisphere_count(enriched))
print("US terr.:   ", us_territory_count(enriched))
print("New Zealand:", new_zealand_count(enriched))

## Analysis Q2 - Closest stations in New Zealand (spherical distance)

The **Haversine** formula gives great-circle distance on a sphere (R = 6371 km), accounting for
curvature and meridian convergence - a planar metric would overstate east-west separation at
41 deg S. Applied to all 15 NZ stations (105 unique pairs).

> **Result (cluster run).** The closest pair is **NZ000093417 (Paraparaumu AWS)** and
> **NZM00093439 (Wellington Aero AWS)** at **~50.5 km** - the minimum across 105 pairs, reflecting
> how sparse the 15-station NZ network is. (Pinned in `tests/test_distance.py`.)

In [ ]:
from ghcn.analysis.station_analysis import nz_stations
from ghcn.analysis.distance import pairwise_distances, closest_pair

nz = nz_stations(enriched)
pairs = pairwise_distances(nz)                  # CROSS JOIN + Haversine UDF, upper triangle only
print("unique pairs:", pairs.count())
print("closest:", closest_pair(pairs))

## Analysis Q3 - The daily summaries in detail

Active stations by `daily` vs `inventory`, the five core-element counts across **all** of daily,
and TMAX observations with no same-day TMIN (an efficient LEFT ANTI JOIN on `[ID, DATE]`).

> **Result (cluster run).** Active in 2026 by `daily` **38,678** vs by `inventory` **37,422**
> (difference **1,256** - `inventory`'s `LASTYEAR` lags recent reporting). Core-element counts over
> all years: **PRCP 1,095,438,345**, **TMAX 466,229,927**, **TMIN 465,041,338**,
> **SNOW 367,671,806**, **SNWD 306,402,150** - PRCP dominates. **10,727,375** TMAX observations
> have no same-day TMIN, from **28,762** stations (~2.3% of TMAX).

In [ ]:
from ghcn.analysis.observations import (
    active_stations_in_year, core_element_observations, tmax_without_tmin,
)
print("active 2026 (daily):    ", active_stations_in_year(daily, 2026))
print("active 2026 (inventory):", count_active_in_year(enriched, 2026))

core_element_observations(daily).show()          # ELEMENT, N_OBSERVATIONS (desc)

_, n_obs, n_stations = tmax_without_tmin(daily)
print("TMAX without TMIN:", n_obs, "from", n_stations, "stations")

## Visualizations Q1 - New Zealand TMIN / TMAX

Filter `daily` to NZ TMIN/TMAX (LEFT SEMI), aggregate to monthly means, smooth with a 12-month
rolling average, and align all 15 stations on a shared axis. Gaps are preserved, never
interpolated.

> **Result (cluster run).** **500,840** NZ TMIN/TMAX observations across **87 years**
> (1940-2026), **9,062** station-months, with **44 dates** on which no NZ station reported. Figures
> rendered on the driver and saved to [`reports/figures/`](../reports/figures).

In [ ]:
from ghcn.visualization.nz_timeseries import (
    nz_tmin_tmax, monthly_station_series, country_monthly_average,
    plot_station_subplots, plot_country_average, plot_nz_station_map,
)
nz_temps = nz_tmin_tmax(daily, nz)
print("NZ TMIN/TMAX observations:", nz_temps.count())

monthly = monthly_station_series(nz_temps)                 # small aggregate -> pandas
plot_nz_station_map(nz)                                     # -> fig3_nz_station_map.png
plot_station_subplots(monthly.toPandas())                  # -> fig4_nz_station_subplots.png
plot_country_average(country_monthly_average(nz_temps).toPandas())  # -> fig5_nz_countrywide.png

![NZ station map](../reports/figures/fig3_nz_station_map.png)

![NZ per-station series](../reports/figures/fig4_nz_station_subplots.png)

## Visualizations Q2 - Global rainfall choropleth (2025)

Average daily rainfall per country-year from **all** PRCP observations (~1.095 B), then a 2025
choropleth on a natural-earth projection with a sequential (blue) scale.

> **Result (cluster run).** **17,789** country-year rows; mean **4.20 mm/day**, max **436.1 mm/day**
> - **Equatorial Guinea 2000**, from a **single** observation (a sparse-sample outlier, investigated
> not deleted). For 2025 there are **172** country-years (mean 5.28, max 54.0 = Azerbaijan from 3
> obs). In the report figure the ISO-3 mapping resolves all 172 (Svalbard/Jan Mayen share `SJM`, so
> the map shows 170), the colour scale is capped at the 95th percentile, and 47 countries with
> stations but no 2025 rainfall are shown as missing rather than zero.
>
> *Note:* `plot_choropleth_plotly` here renders the base map (natural earth, sequential scale) and
> reports unmatched countries; the 95th-percentile display cap and `SJM` de-duplication used for the
> report figure are applied in the report's build (a natural enhancement to fold into the function).

In [ ]:
from ghcn.analysis.rainfall import (
    average_daily_rainfall_by_year_country, rainfall_descriptive_stats,
    highest_rainfall_year_country, rainfall_for_year,
)
rainfall = average_daily_rainfall_by_year_country(daily, meta["stations"])
print("country-year rows:", rainfall.count())
rainfall_descriptive_stats(rainfall).show()      # summary of AVG_DAILY_PRCP_MM
print("highest:", highest_rainfall_year_country(rainfall))  # incl. N_OBSERVATIONS

In [ ]:
from ghcn.visualization.choropleth import plot_choropleth_plotly

r2025 = rainfall_for_year(rainfall, 2025).toPandas()        # 172 country-years (small)
fig, matched, unmatched = plot_choropleth_plotly(
    r2025,
    output_html="../reports/figures/rainfall_2025_choropleth.html",
    output_png="../reports/figures/fig6_rainfall_choropleth_2025.png",
    year=2025,
)
print("countries matched to ISO-3:", matched['ISO3'].nunique(),
      "| unmatched:", len(unmatched))

![2025 rainfall choropleth](../reports/figures/fig6_rainfall_choropleth_2025.png)

## Wrap-up

Key results - 132,501 stations (20,505 with all five core elements, 16,189 precipitation-only,
63 never in `daily`); closest NZ pair ~50.5 km; PRCP dominates with 1.095 B observations; the
2025 rainfall map is right-skewed and coverage-limited. Full discussion and figures:
[`reports/final_report.md`](../reports/final_report.md); the cluster-rendered outputs are in
[`docs/executed_notebooks/`](../docs/executed_notebooks).

In [ ]:
spark.stop()